# Obtaining the cutouts of the plates

In [1]:
import json
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import random
import base64
import gzip
import requests
import os

##

In [2]:
with open("../Step_2_Integration/observability_bright.json", "r") as f:
    data = json.load(f)

In [3]:
def retrieve_and_save(plate_id, sol_id, ra, dec, path, mpcnum):
    os.makedirs(path, exist_ok=True)

    url = "https://api.starglass.cfa.harvard.edu/public/dasch/dr7/cutout"
    payload = {
        "plate_id": plate_id,
        "solution_number": int(sol_id),
        "center_ra_deg": float(ra),
        "center_dec_deg": float(dec),
    }

    r = requests.post(url, json=payload, headers={"Accept": "application/json"})
    r.raise_for_status()

    fits_bytes = gzip.decompress(base64.b64decode(r.json()))

    filename = f"{mpcnum}_{plate_id}.fits"
    filepath = os.path.join(path, filename)

    with open(filepath, "wb") as f:
        f.write(fits_bytes)

    return filepath

In [4]:
def get_all_files(mpcnum, cutoff=None, sleep=10, max_rounds=None):
    subdata = np.asarray(data[str(int(mpcnum))])
    ra = subdata[:, 0].astype(float)
    dec = subdata[:, 1].astype(float)
    plate_id, sol_id = np.array([entry.split(":") for entry in subdata[:, 7]]).T

    if cutoff is None:
        cutoff = len(plate_id)

    os.makedirs(str(mpcnum), exist_ok=True)
    todo = list(range(cutoff))
    round_num = 0

    while todo and (max_rounds is None or round_num <= max_rounds):
        failed = []
        print(f"\nRound {round_num}: trying {len(todo)} files")

        for i in todo:
            filename = f"{mpcnum}_{plate_id[i]}.fits"
            filepath = os.path.join(str(mpcnum), filename)

            if os.path.exists(filepath):
                print(f"Skipping {i}: {filename}")
                continue

            try:
                print(f"{mpcnum} {i}: {ra[i]} {dec[i]} {plate_id[i]} {sol_id[i]}")
                retrieve_and_save(plate_id[i], sol_id[i], ra[i], dec[i], str(mpcnum), str(mpcnum))
            except Exception as e:
                print(f"Failed {mpcnum} {i}: {e}")
                failed.append(i)

        todo = failed
        round_num += 1
        if todo:
            time.sleep(sleep)

    print("All files retrieved." if not todo else f"Still failed: {todo}")

In [5]:
import time

mpc_ids = [1, 29, 196, 702, 420, 624, 4709, 3978, 15436]
timings = []

t0 = time.perf_counter()

for i, num in enumerate(mpc_ids, start=1):
    start = time.perf_counter()
    get_all_files(num, None)
    elapsed = time.perf_counter() - start
    total_elapsed = time.perf_counter() - t0

    timings.append((num, elapsed, total_elapsed))
    print(f"{i:>3}/{len(mpc_ids)}  MPC {num:<6}  step={elapsed:8.2f}s  total={total_elapsed/60:8.2f} min")


Round 0: trying 6263 files
Skipping 0: 1_i00768.fits
Skipping 1: 1_i01460.fits
Skipping 2: 1_b06327.fits
Skipping 3: 1_b06363.fits
Skipping 4: 1_b06387.fits
Skipping 5: 1_b06388.fits
Skipping 6: 1_b06561.fits
Skipping 7: 1_i07633.fits
Skipping 8: 1_i07647.fits
Skipping 9: 1_i07678.fits
Skipping 10: 1_i07723.fits
Skipping 11: 1_i07825.fits
Skipping 12: 1_i07880.fits
Skipping 13: 1_a00379.fits
Skipping 14: 1_i11021.fits
Skipping 15: 1_b13539.fits
Skipping 16: 1_b13580.fits
Skipping 17: 1_b13728.fits
Skipping 18: 1_b13936.fits
Skipping 19: 1_b14045.fits
Skipping 20: 1_b14745.fits
Skipping 21: 1_b15043.fits
Skipping 22: 1_b15954.fits
Skipping 23: 1_b15996.fits
Skipping 24: 1_b16333.fits
Skipping 25: 1_b16462.fits
Skipping 26: 1_b16839.fits
Skipping 27: 1_b16924.fits
Skipping 28: 1_i15663.fits
Skipping 29: 1_i15821.fits
Skipping 30: 1_i15911.fits
Skipping 31: 1_b17469.fits
Skipping 32: 1_b17575.fits
Skipping 33: 1_b17594.fits
Skipping 34: 1_b17637.fits
Skipping 35: 1_i16201.fits
Skipping 3


KeyboardInterrupt



In [ ]:
print("DONE")